# 1. Closed Queuing network simulation (17 pt)
Build a simulator to capture the following system and answer the questions below. A total of N=40 jobs circulate in different parts of the systems: CPU, disk, and resting area. There is one CPU and two disks, one slow and one fast. Each job starts at the CPU station and takes an average of 2 seconds. After CPU, a job fetches the data from the disk, needing an average of 3000 disk cycles. There are two possible choices of disks, fast and slow, which have a speed of 1000 cycles per second and 100 cycles per seconds, respectively. After the disk station, the job can rest for 15 seconds on average. The figure below gives an overview of the system:


![image](Queued.png)

The choices of distributions are up to you. You are welcomed to try different distributions to answer the following questions and assess the impact of different variants.


Questions:
(3p) What will be the maximal system throughput in terms of number of jobs, given two different load balancing strategies for sending jobs between the fast and slow disk? You need to come up with two load balancing strategies and compare them. 
(3p) What will the system throughput and average response time of a job be if a faster CPU is used? Say, CPU time is reduced to 1 seconds on average and the rest of the system remains the same.
(3p) What will the system throughput and average response time of a job be if a second fast disk is added? Again, you need to compare the throughput under two different load balancing strategies.
(3p) What will the system throughput and average response time of a job be if a faster CPU is used and a second fast disk is added? And, you use the better load balancing strategies out of two you propose in the first question?
(5pt) If you can answer all the above questions with different number of N and plot them in the following style:


![image](Graph.png)

In [17]:
# imports
import os
import sys
from contextlib import contextmanager, redirect_stdout
from typing import Generator

import matplotlib.pyplot as plt
import numpy as np
import simpy
from matplotlib.axes import Axes
from matplotlib.figure import Figure

RNG = np.random.default_rng(seed=0)

@contextmanager
def suppress_print():
    """Temporarily disable print statements."""
    with redirect_stdout(open(os.devnull, "w")):
        yield

In [ ]:
# Basic Closed Queueing Network Simulation
class ClosedQueueNetwork:
    def __init__(self, N_jobs=40):
        self.env = simpy.Environment()
        self.cpu = simpy.Resource(self.env, capacity=1)
        self.slow_disk = simpy.Resource(self.env, capacity=1)
        self.fast_disk = simpy.Resource(self.env, capacity=1)
        self.N_jobs = N_jobs

    def job_process(self, job_id):
        while True:
            # CPU phase
            with self.cpu.request() as req:
                yield req
                print(f"Time {self.env.now:.2f}: job {job_id} using CPU")
                yield self.env.timeout(np.random.exponential(2))  # mean 2s

            # Disk phase: randomly select fast or slow
            if np.random.rand() < 0.5:
                # Fast disk
                with self.fast_disk.request() as req:
                    yield req
                    disk_time = 3000 / 1000  # 3s on avg.
                    yield self.env.timeout(np.random.exponential(disk_time))
            else:
                # Slow disk
                with self.slow_disk.request() as req:
                    yield req
                    disk_time = 3000 / 100   # 30s on avg.
                    yield self.env.timeout(np.random.exponential(disk_time))
            
            # Resting
            print(f"Time {self.env.now:.2f}: job {job_id} resting")
            yield self.env.timeout(np.random.exponential(15))

    def run(self, until=1000):
        for i in range(self.N_jobs):
            self.env.process(self.job_process(i))
        self.env.run(until=until)


In [19]:
ClosedQueueNetwork(N_jobs=40).run(until=1000)

Time 0.00: job 0 using CPU
Time 0.76: job 1 using CPU
Time 2.18: job 2 using CPU
Time 4.47: job 3 using CPU
Time 4.84: job 0 resting
Time 7.82: job 4 using CPU
Time 8.34: job 5 using CPU
Time 8.46: job 6 using CPU
Time 9.49: job 7 using CPU
Time 10.40: job 8 using CPU
Time 12.69: job 9 using CPU
Time 13.13: job 1 resting
Time 13.75: job 10 using CPU
Time 14.01: job 2 resting
Time 14.14: job 11 using CPU
Time 14.16: job 12 using CPU
Time 15.05: job 13 using CPU
Time 15.95: job 3 resting
Time 17.10: job 14 using CPU
Time 17.83: job 15 using CPU
Time 18.29: job 16 using CPU
Time 20.54: job 7 resting
Time 21.40: job 17 using CPU
Time 21.55: job 18 using CPU
Time 21.56: job 19 using CPU
Time 22.05: job 4 resting
Time 22.67: job 20 using CPU
Time 28.55: job 21 using CPU
Time 29.27: job 22 using CPU
Time 31.27: job 23 using CPU
Time 32.41: job 10 resting
Time 33.78: job 24 using CPU
Time 36.46: job 25 using CPU
Time 37.05: job 26 using CPU
Time 38.08: job 12 resting
Time 38.60: job 14 resting